In [ ]:
import math
from collections import defaultdict
from pathlib import Path
from typing import Optional

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import pandas as pd
import yaml
from matplotlib.font_manager import FontProperties
from matplotlib.lines import Line2D

In [ ]:
mpl.style.use("../../mystyle.mpl")

In [ ]:
FIG_DIR = Path("figs/")
FIG_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = Path("../../results/ssn")

# Loading helpers

In [ ]:
def _get_nested(d: dict, keys: list[str]):
    """Retrieve a value from a nested dict using a list of keys."""
    for k in keys:
        d = d[k]
    return d


def load_runs(results_dir: Path, noise_param: str | list[str]) -> dict:
    """Load all runs from results_dir, grouped by one noise-level parameter.

    Args:
        results_dir: Path to results/ssn/<exp_type>/
        noise_param: Key or list of nested keys into the params dict
                     for the noise level, e.g. "init_noise" or
                     ["stdp", "ws", "noise"].

    Returns:
        {"dkl":   {noise_level: DataFrame},
         "asym":  {noise_level: DataFrame},
         "params": dict}
        DataFrames have shape (num_val_steps, num_runs);
        columns are named "<sweep_id>_<seed_id>".
    """
    if isinstance(noise_param, str):
        noise_param = [noise_param]

    dkl_data: dict[float, dict] = {}
    asym_data: dict[float, dict] = {}
    params_cache: dict[str, dict] = {}
    result_params = None

    for npz_path in sorted(results_dir.glob("*.npz")):
        stem = npz_path.stem  # e.g. "00_03"
        sweep_id, seed_id = stem.split("_", 1)

        if sweep_id not in params_cache:
            with open(results_dir / f"{sweep_id}_params.yaml") as f:
                params_cache[sweep_id] = yaml.safe_load(f)
        p = params_cache[sweep_id]
        if result_params is None:
            result_params = p

        noise_val = round(float(_get_nested(p, noise_param)), 10)

        data = np.load(npz_path)
        dkl_data.setdefault(noise_val, {})[stem] = data["dkls"]
        asym_data.setdefault(noise_val, {})[stem] = data["all_asym"]

    return {
        "dkl": {n: pd.DataFrame(v) for n, v in sorted(dkl_data.items())},
        "asym": {n: pd.DataFrame(v) for n, v in sorted(asym_data.items())},
        "params": result_params,
    }


def load_kp_runs(
    results_dir: Path,
    noise_param: str | list[str],
    lr_param: str | list[str],
) -> dict:
    """Load KP-sweep runs, grouped by (noise_level, kp_lr).

    Returns:
        {noise_level: {"dkl":  {kp_lr: DataFrame},
                       "asym": {kp_lr: DataFrame},
                       "params": dict}}
    """
    if isinstance(noise_param, str):
        noise_param = [noise_param]
    if isinstance(lr_param, str):
        lr_param = [lr_param]

    store: dict = defaultdict(lambda: {"dkl": {}, "asym": {}, "params": None})
    params_cache: dict[str, dict] = {}

    for npz_path in sorted(results_dir.glob("*.npz")):
        stem = npz_path.stem
        sweep_id, seed_id = stem.split("_", 1)

        if sweep_id not in params_cache:
            with open(results_dir / f"{sweep_id}_params.yaml") as f:
                params_cache[sweep_id] = yaml.safe_load(f)
        p = params_cache[sweep_id]

        noise_val = round(float(_get_nested(p, noise_param)), 10)
        lr_val = round(float(_get_nested(p, lr_param)), 15)

        if store[noise_val]["params"] is None:
            store[noise_val]["params"] = p

        data = np.load(npz_path)
        store[noise_val]["dkl"].setdefault(lr_val, {})[stem] = data["dkls"]
        store[noise_val]["asym"].setdefault(lr_val, {})[stem] = data["all_asym"]

    return {
        noise: {
            "dkl": {lr: pd.DataFrame(v) for lr, v in sorted(d["dkl"].items())},
            "asym": {lr: pd.DataFrame(v) for lr, v in sorted(d["asym"].items())},
            "params": d["params"],
        }
        for noise, d in sorted(store.items())
    }

# Plotting helpers

In [ ]:
def add_stats(df, regex=r"^\d"):
    df["mean"] = df.filter(regex=regex).mean(axis=1)
    df["std"] = df.filter(regex=regex).std(axis=1)
    df["median"] = df.filter(regex=regex).quantile(q=0.5, axis=1)
    df["lower_q"] = df.filter(regex=regex).quantile(q=0.25, axis=1)
    df["upper_q"] = df.filter(regex=regex).quantile(q=0.75, axis=1)


def plot_epochs(
    ax,
    dat: pd.DataFrame,
    epochs: npt.NDArray,
    yscale: str = "log",
    label="",
    **kwargs,
):
    p = ax.plot(epochs, dat["median"], label=label, **kwargs)
    fill_between(
        ax,
        epochs,
        y1=dat["lower_q"],
        y2=dat["upper_q"],
        color=p[0].get_color(),
        alpha=0.3,
    )
    ax.set_yscale(yscale)
    return ax


def ebar(
    ax,
    dat: dict[float, npt.NDArray],
    pos: int,
    color: str,
    label: Optional[str] = None,
    **kwargs,
):
    u_err = dat["median"].iloc[-1] - dat["lower_q"].iloc[-1]
    l_err = dat["upper_q"].iloc[-1] - dat["median"].iloc[-1]
    ax.errorbar(
        [pos],
        [dat["median"].iloc[-1]],
        yerr=np.array([[u_err], [l_err]]),
        capsize=4,
        color=color,
        label=label,
        **kwargs,
    )
    return ax


def plot_baseline(ax, dat, color, kwargs_line={}, kwargs_fill={}):
    ax.axhline(y=dat["median"].iloc[-1], color=color, zorder=-1, **kwargs_line)
    x = np.array([ax.get_xlim()[0], ax.get_xlim()[1]])
    fill_between(
        ax,
        x,
        y1=dat["lower_q"].iloc[-1],
        y2=dat["upper_q"].iloc[-1],
        color=color,
        alpha=0.3,
        **kwargs_fill,
    )
    ax.set_xlim(x[0], x[1])


def plot_roundmarker(ax, x, y, char, color):
    circle = mpl.markers.MarkerStyle("o", fillstyle="none").scaled(1.0)
    ax.plot([x], [y], marker=circle, color=color, markersize=8)
    ax.text(x, y, f"{char}", color=color, ha="center", va="center", fontsize="x-small")


def axis_breaker(ax, y_pos, breaker_width=10, breaker_dist=0.02, whitespace_size=5):
    d = 0.5
    kwargs = dict(
        marker=[(-1, -d), (1, d)],
        markersize=breaker_width,
        linestyle="none",
        color="k",
        mec="k",
        mew=1,
        clip_on=False,
    )
    ax.plot(
        [0.0, 1],
        [y_pos - breaker_dist / 2] * 2,
        transform=ax.transAxes,
        **kwargs,
        zorder=120,
    )
    ax.plot(
        [0.0, 1],
        [y_pos + breaker_dist / 2] * 2,
        transform=ax.transAxes,
        **kwargs,
        zorder=110,
    )
    ax.plot(
        [0.0, 1],
        [y_pos] * 2,
        transform=ax.transAxes,
        marker="o",
        color="white",
        clip_on=False,
        linestyle="none",
        ms=whitespace_size,
        zorder=100,
    )


def fill_between(ax, *args, **kwargs):
    kwargs.setdefault("linewidth", 0.0)
    return ax.fill_between(*args, **kwargs)


def extract_data(df_dict):
    """Extract final-step median and IQR errors from a dict of DataFrames."""
    median = np.array([df["median"].iloc[-1] for df in df_dict.values()])
    upper_q = np.array([df["upper_q"].iloc[-1] for df in df_dict.values()])
    lower_q = np.array([df["lower_q"].iloc[-1] for df in df_dict.values()])
    u_err = median - lower_q
    l_err = upper_q - median
    return median, np.array([u_err, l_err])

In [ ]:
STR_INIT_NOISE = r"$\sigma^\mathrm{noise}_\mathrm{syn}$"
STR_PLAST_NOISE = r"$\sigma^\mathrm{noise}_\mathrm{plast}$"

# Synaptic noise

## Without SAL

In [ ]:
NOISE_LEVELS = [0.0, 0.2, 0.4, 0.6, 0.8]

# no_sal = load_runs(RESULTS_DIR / "syn_noise", "init_noise")
no_sal = load_runs(RESULTS_DIR / "syn_noise_test", "init_noise")

In [ ]:
no_sal["dkl"]

In [ ]:
for df in no_sal["dkl"].values():
    add_stats(df)

In [ ]:
for df in no_sal["asym"].values():
    add_stats(df)

In [ ]:
no_sal["dkl"]

## With SAL

In [ ]:
w_sal = load_runs(RESULTS_DIR / "syn_noise_sal", "init_noise")

for df in w_sal["dkl"].values():
    add_stats(df)
for df in w_sal["asym"].values():
    add_stats(df)

## With Kolen-Pollack

In [ ]:
kp = load_runs(RESULTS_DIR / "syn_noise_kp", "init_noise")

for df in kp["dkl"].values():
    add_stats(df)
for df in kp["asym"].values():
    add_stats(df)

## Plot

In [ ]:
EPOCHS = np.arange(0, no_sal["params"]["num_epochs"], no_sal["params"]["val_step"])
NOISE_ID = 4

fig, ax = plt.subplots(
    2,
    2,
    sharex="col",
    figsize=(18 / 2.54, 8 / 2.54),
    squeeze=False,
    gridspec_kw={"top": 0.8, "wspace": 0.4, "right": 0.98, "bottom": 0.0},
)

plot_epochs(
    ax[0, 0],
    no_sal["asym"][0.0],
    EPOCHS,
    label=f"baseline ({STR_INIT_NOISE} = 0, {STR_PLAST_NOISE} = 0)",
    yscale="linear",
    linestyle="dashed",
)
plot_epochs(
    ax[0, 0],
    no_sal["asym"][NOISE_LEVELS[NOISE_ID]],
    EPOCHS,
    label="w/o SAL",
    yscale="linear",
    linestyle="dotted",
)
plot_epochs(
    ax[0, 0],
    w_sal["asym"][NOISE_LEVELS[NOISE_ID]],
    EPOCHS,
    label="with SAL",
    yscale="linear",
)
plot_epochs(
    ax[0, 0],
    kp["asym"][NOISE_LEVELS[NOISE_ID]],
    EPOCHS,
    label="with KP",
    yscale="linear",
    linestyle="dashdot",
)

plot_epochs(
    ax[1, 0],
    no_sal["dkl"][0.0],
    EPOCHS,
    label=f"baseline ({STR_INIT_NOISE} = 0, {STR_PLAST_NOISE} = 0)",
    linestyle="dashed",
)
plot_epochs(
    ax[1, 0],
    no_sal["dkl"][NOISE_LEVELS[NOISE_ID]],
    EPOCHS,
    label="w/o SAL",
    linestyle="dotted",
)
plot_epochs(ax[1, 0], w_sal["dkl"][NOISE_LEVELS[NOISE_ID]], EPOCHS, label="with SAL")
plot_epochs(
    ax[1, 0],
    kp["dkl"][NOISE_LEVELS[NOISE_ID]],
    EPOCHS,
    label="with KP",
    linestyle="dashdot",
)

axis_breaker(ax[0, 0], 0.18, breaker_dist=0.04, breaker_width=6, whitespace_size=2.5)
axis_breaker(ax[0, 1], 0.18, breaker_dist=0.04, breaker_width=6, whitespace_size=2.5)

ax[1, 0].set_xlim(right=EPOCHS[-1] * 1.28)

LEFT = EPOCHS[-1] * 1.09
RIGHT = EPOCHS[-1] * 1.21
plot_roundmarker(
    ax[0, 0], LEFT, no_sal["asym"][0.0]["median"].iloc[-1], char="1", color="C0"
)
plot_roundmarker(
    ax[0, 0],
    LEFT,
    no_sal["asym"][NOISE_LEVELS[NOISE_ID]]["median"].iloc[-1],
    char="2",
    color="C1",
)
plot_roundmarker(
    ax[0, 0],
    LEFT,
    w_sal["asym"][NOISE_LEVELS[NOISE_ID]]["median"].iloc[-1],
    char="3",
    color="C2",
)
plot_roundmarker(
    ax[0, 0],
    LEFT,
    kp["asym"][NOISE_LEVELS[NOISE_ID]]["median"].iloc[-1],
    char="4",
    color="C3",
)

plot_roundmarker(
    ax[1, 0], LEFT, no_sal["dkl"][0.0]["median"].iloc[-1], char="1", color="C0"
)
plot_roundmarker(
    ax[1, 0],
    LEFT,
    no_sal["dkl"][NOISE_LEVELS[NOISE_ID]]["median"].iloc[-1],
    char="2",
    color="C1",
)
plot_roundmarker(
    ax[1, 0],
    RIGHT,
    w_sal["dkl"][NOISE_LEVELS[NOISE_ID]]["median"].iloc[-1] - 0.0007,
    char="3",
    color="C2",
)
plot_roundmarker(
    ax[1, 0],
    RIGHT,
    kp["dkl"][NOISE_LEVELS[NOISE_ID]]["median"].iloc[-1] + 0.0007,
    char="4",
    color="C3",
)

DELTA = 0.02
for n in NOISE_LEVELS:
    ebar(ax[0, 1], w_sal["asym"][n], n + DELTA, color="C2", marker="x", label="")
    ebar(ax[0, 1], no_sal["asym"][n], n, color="C1", marker="3", label="")
    ebar(ax[0, 1], kp["asym"][n], n - DELTA, color="C3", marker="4", label="")
    ebar(ax[1, 1], w_sal["dkl"][n], n + DELTA, color="C2", marker="x", label="")
    ebar(ax[1, 1], no_sal["dkl"][n], n, color="C1", marker="3", label="")
    ebar(ax[1, 1], kp["dkl"][n], n - DELTA, color="C3", marker="4", label="")

# store reference values for KP-lr-sweep comparison
ref_syn_noise_sal = {
    0.4: (
        w_sal["dkl"][0.4]["median"].iloc[-1],
        w_sal["dkl"][0.4]["lower_q"].iloc[-1],
        w_sal["dkl"][0.4]["upper_q"].iloc[-1],
    ),
    0.8: (
        w_sal["dkl"][0.8]["median"].iloc[-1],
        w_sal["dkl"][0.8]["lower_q"].iloc[-1],
        w_sal["dkl"][0.8]["upper_q"].iloc[-1],
    ),
}

plot_baseline(ax[0, 1], no_sal["asym"][0.0], "C0", kwargs_line={"linestyle": "dashed"})
plot_baseline(ax[1, 1], no_sal["dkl"][0.0], "C0", kwargs_line={"linestyle": "dashed"})

plot_roundmarker(ax[0, 1], 0.0, 5.0e-5, char="1", color="C0")
plot_roundmarker(ax[0, 1], 0.75, 0.8, char="2", color="C1")
plot_roundmarker(ax[0, 1], 0.78, 1.0e-3, char="3", color="C2")
plot_roundmarker(ax[0, 1], 0.74, 0.08, char="4", color="C3")

plot_roundmarker(ax[1, 1], 0.0, 6.0e-3, char="1", color="C0")
plot_roundmarker(ax[1, 1], 0.75, 1.5e-2, char="2", color="C1")
plot_roundmarker(ax[1, 1], 0.78, 6.0e-3, char="3", color="C2")
plot_roundmarker(ax[1, 1], 0.72, 8.0e-3, char="4", color="C3")

VAR_YLIM = (-0.5e-4, 3)
ax[1, 0].set_xlabel("wake-sleep cycles")
ax[1, 0].ticklabel_format(style="sci", scilimits=(-3, 4), axis="x")
ax[0, 0].set_ylabel(r"$\mathrm{Var}\, (W_{ij} - W_{ji})$")
ax[0, 0].set_yscale("symlog", linthresh=1e-4)
ax[0, 0].set_ylim(*VAR_YLIM)
ax[1, 0].set_ylabel(r"$D_\mathrm{KL}(p \| p^*)$")
ax[1, 0].set_ylim(0.1e-2, 2)
ax[0, 0].text(
    0.05,
    0.2,
    STR_INIT_NOISE + f" = {NOISE_LEVELS[NOISE_ID]}",
    transform=ax[0, 0].transAxes,
    ha="left",
    va="top",
    size=8,
    bbox=dict(facecolor=(1, 1, 1, 0.7), edgecolor="black", boxstyle="round"),
)
ax[1, 0].text(
    0.575,
    -0.275,
    STR_INIT_NOISE + f" = {NOISE_LEVELS[NOISE_ID]}",
    transform=ax[0, 0].transAxes,
    ha="left",
    va="top",
    size=8,
    bbox=dict(facecolor=(1, 1, 1, 0.7), edgecolor="black", boxstyle="round"),
)

ax[1, 1].set_yscale("log")
ax[1, 1].yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
ax[1, 1].margins(y=0.2)
ax[1, 1].set_xlabel(STR_INIT_NOISE)
ax[1, 1].set_ylabel(r"$D_\mathrm{KL}(p \| p^*)$")
ax[0, 1].set_ylabel(r"$\mathrm{Var}\, (W_{ij} - W_{ji})$")
ax[0, 1].set_xticks(NOISE_LEVELS)
ax[0, 1].set_yscale("symlog", linthresh=1e-4)
ax[0, 1].set_ylim(*VAR_YLIM)
ax[1, 1].set_ylim([4e-3, 2.1e-2])

handles_1, labels = ax[1, 0].get_legend_handles_labels()
markers = [None, "3", "x", "4"]
handles = [
    Line2D([], [], color=h1.get_color(), linestyle=h1.get_linestyle(), marker=m)
    for h1, m in zip(handles_1, markers)
]
legend = fig.legend(
    handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.98), ncol=4
)
font_props = FontProperties(weight="bold", size=12)
legend.set_title("synaptic noise", prop=font_props)

In [ ]:
fig.savefig(FIG_DIR / "ssn_initnoise.png", bbox_inches="tight", dpi=300)
fig.savefig(FIG_DIR / "ssn_initnoise.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "ssn_initnoise.svg", bbox_inches="tight")

# Plasticity Noise

## Without SAL

In [ ]:
NOISE_LEVELS = [0.0, 0.2, 0.4, 0.6, 0.8]

no_sal = load_runs(RESULTS_DIR / "plast_noise", ["stdp", "ws", "noise"])

for df in no_sal["dkl"].values():
    add_stats(df)
for df in no_sal["asym"].values():
    add_stats(df)

## With SAL

In [ ]:
w_sal = load_runs(RESULTS_DIR / "plast_noise_sal", ["stdp", "ws", "noise"])

for df in w_sal["dkl"].values():
    add_stats(df)
for df in w_sal["asym"].values():
    add_stats(df)

## With Kolen-Pollack

In [ ]:
kp = load_runs(RESULTS_DIR / "plast_noise_kp", ["stdp", "ws", "noise"])

for df in kp["dkl"].values():
    add_stats(df)
for df in kp["asym"].values():
    add_stats(df)

## Plot

In [ ]:
EPOCHS = np.arange(0, no_sal["params"]["num_epochs"], no_sal["params"]["val_step"])
NOISE_ID = 4

fig, ax = plt.subplots(
    2,
    2,
    sharex="col",
    figsize=(18 / 2.54, 8 / 2.54),
    squeeze=False,
    gridspec_kw={"top": 0.8, "wspace": 0.4, "right": 0.98, "bottom": 0.0},
)

plot_epochs(
    ax[0, 0],
    no_sal["asym"][0.0],
    EPOCHS,
    label=f"baseline ({STR_INIT_NOISE} = 0.2, {STR_PLAST_NOISE} = 0)",
    yscale="linear",
    linestyle="dashed",
    c="purple",
)
plot_epochs(
    ax[0, 0],
    no_sal["asym"][NOISE_LEVELS[NOISE_ID]],
    EPOCHS,
    label="w/o SAL",
    yscale="linear",
    linestyle="dotted",
    c="C1",
)
plot_epochs(
    ax[0, 0],
    w_sal["asym"][NOISE_LEVELS[NOISE_ID]],
    EPOCHS,
    label="with SAL",
    yscale="linear",
    c="C2",
)
plot_epochs(
    ax[0, 0],
    kp["asym"][NOISE_LEVELS[NOISE_ID]],
    EPOCHS,
    label="with KP",
    yscale="linear",
    linestyle="dashdot",
    c="C3",
)

plot_epochs(
    ax[1, 0],
    no_sal["dkl"][0.0],
    EPOCHS,
    label=f"baseline ({STR_INIT_NOISE} = 0.2, {STR_PLAST_NOISE} = 0)",
    linestyle="dashed",
    c="purple",
)
plot_epochs(
    ax[1, 0],
    no_sal["dkl"][NOISE_LEVELS[NOISE_ID]],
    EPOCHS,
    label="w/o SAL",
    linestyle="dotted",
    c="C1",
)
plot_epochs(
    ax[1, 0], w_sal["dkl"][NOISE_LEVELS[NOISE_ID]], EPOCHS, label="with SAL", c="C2"
)
plot_epochs(
    ax[1, 0],
    kp["dkl"][NOISE_LEVELS[NOISE_ID]],
    EPOCHS,
    label="with KP",
    linestyle="dashdot",
    c="C3",
)

ax[1, 0].set_xlim(right=EPOCHS[-1] * 1.28)

LEFT = EPOCHS[-1] * 1.09
RIGHT = EPOCHS[-1] * 1.21
plot_roundmarker(
    ax[0, 0], LEFT, no_sal["asym"][0.0]["median"].iloc[-1], char="1", color="purple"
)
plot_roundmarker(
    ax[0, 0],
    LEFT,
    no_sal["asym"][NOISE_LEVELS[NOISE_ID]]["median"].iloc[-1],
    char="2",
    color="C1",
)
plot_roundmarker(
    ax[0, 0],
    LEFT,
    w_sal["asym"][NOISE_LEVELS[NOISE_ID]]["median"].iloc[-1],
    char="3",
    color="C2",
)
plot_roundmarker(
    ax[0, 0],
    RIGHT,
    kp["asym"][NOISE_LEVELS[NOISE_ID]]["median"].iloc[-1],
    char="4",
    color="C3",
)

plot_roundmarker(
    ax[1, 0], LEFT, no_sal["dkl"][0.0]["median"].iloc[-1], char="1", color="purple"
)
plot_roundmarker(
    ax[1, 0],
    LEFT,
    no_sal["dkl"][NOISE_LEVELS[NOISE_ID]]["median"].iloc[-1],
    char="2",
    color="C1",
)
plot_roundmarker(
    ax[1, 0],
    RIGHT,
    w_sal["dkl"][NOISE_LEVELS[NOISE_ID]]["median"].iloc[-1] - 0.0007,
    char="3",
    color="C2",
)
plot_roundmarker(
    ax[1, 0],
    RIGHT,
    kp["dkl"][NOISE_LEVELS[NOISE_ID]]["median"].iloc[-1] + 0.0007,
    char="4",
    color="C3",
)

DELTA = 0.02
for n in NOISE_LEVELS:
    ebar(ax[0, 1], w_sal["asym"][n], n + DELTA, color="C2", marker="x", label="")
    ebar(ax[0, 1], no_sal["asym"][n], n, color="C1", marker="3", label="")
    ebar(ax[0, 1], kp["asym"][n], n - DELTA, color="C3", marker="4", label="")
    ebar(ax[1, 1], w_sal["dkl"][n], n + DELTA, color="C2", marker="x", label="")
    ebar(ax[1, 1], no_sal["dkl"][n], n, color="C1", marker="3", label="")
    ebar(ax[1, 1], kp["dkl"][n], n - DELTA, color="C3", marker="4", label="")

ref_plast_noise_sal = {
    0.4: (
        w_sal["dkl"][0.4]["median"].iloc[-1],
        w_sal["dkl"][0.4]["lower_q"].iloc[-1],
        w_sal["dkl"][0.4]["upper_q"].iloc[-1],
    ),
    0.8: (
        w_sal["dkl"][0.8]["median"].iloc[-1],
        w_sal["dkl"][0.8]["lower_q"].iloc[-1],
        w_sal["dkl"][0.8]["upper_q"].iloc[-1],
    ),
}

plot_baseline(
    ax[0, 1], no_sal["asym"][0.0], "purple", kwargs_line={"linestyle": "dashed"}
)
plot_baseline(
    ax[1, 1], no_sal["dkl"][0.0], "purple", kwargs_line={"linestyle": "dashed"}
)

plot_roundmarker(ax[0, 1], 0.0, 0.25, char="1", color="purple")
plot_roundmarker(ax[0, 1], 0.73, 0.7, char="2", color="C1")
plot_roundmarker(ax[0, 1], 0.75, 1.0e-3, char="3", color="C2")
plot_roundmarker(ax[0, 1], 0.83, 0.15, char="4", color="C3")

plot_roundmarker(ax[1, 1], 0.04, 6.5e-3, char="1", color="purple")
plot_roundmarker(ax[1, 1], 0.84, 0.96e-2, char="2", color="C1")
plot_roundmarker(ax[1, 1], 0.78, 6.0e-3, char="3", color="C2")
plot_roundmarker(ax[1, 1], 0.73, 1.13e-2, char="4", color="C3")

ax[1, 0].set_xlabel("wake-sleep cycles")
ax[1, 0].ticklabel_format(style="sci", scilimits=(-3, 4), axis="x")
ax[0, 0].set_ylabel(r"$\mathrm{Var}\, (W_{ij} - W_{ji})$")
ax[0, 0].set_yscale("log")
ax[1, 0].set_ylabel(r"$D_\mathrm{KL}(p \| p^*)$")
ax[0, 0].text(
    0.05,
    0.2,
    STR_PLAST_NOISE + f" = {NOISE_LEVELS[NOISE_ID]}",
    transform=ax[0, 0].transAxes,
    ha="left",
    va="top",
    size=8,
    bbox=dict(facecolor=(1, 1, 1, 0.7), edgecolor="black", boxstyle="round"),
)
ax[1, 0].text(
    0.575,
    -0.275,
    STR_PLAST_NOISE + f" = {NOISE_LEVELS[NOISE_ID]}",
    transform=ax[0, 0].transAxes,
    ha="left",
    va="top",
    size=8,
    bbox=dict(facecolor=(1, 1, 1, 0.7), edgecolor="black", boxstyle="round"),
)

ax[1, 1].set_yscale("log")
ax[1, 1].yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
ax[1, 1].margins(y=0.2)
ax[1, 1].set_xlabel(STR_PLAST_NOISE)
ax[1, 1].set_ylabel(r"$D_\mathrm{KL}(p \| p^*)$")
ax[0, 1].set_ylabel(r"$\mathrm{Var}\, (W_{ij} - W_{ji})$")
ax[0, 1].set_xticks(NOISE_LEVELS)
ax[0, 1].set_yscale("log")
ax[0, 1].margins(y=0.05)

handles_1, labels = ax[1, 0].get_legend_handles_labels()
markers = [None, "3", "x", "4"]
handles = [
    Line2D([], [], color=h1.get_color(), linestyle=h1.get_linestyle(), marker=m)
    for h1, m in zip(handles_1, markers)
]
legend = fig.legend(
    handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.98), ncol=4
)
font_props = FontProperties(weight="bold", size=12)
legend.set_title("plasticity noise", prop=font_props)

In [ ]:
fig.savefig(FIG_DIR / "ssn_stdpnoise.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "ssn_stdpnoise.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "ssn_stdpnoise.svg", bbox_inches="tight")

# Kolen-Pollack weight decay rate

## Synaptic noise

In [ ]:
noise_levels = [0.4, 0.8]
kp_lr_syn = 1e-6 * np.array([1, 2, 4, 8, 16, 32, 64, 120, 240, 480])
print(kp_lr_syn)

kp_data_syn = load_kp_runs(
    RESULTS_DIR / "syn_noise_kp_lr",
    noise_param="init_noise",
    lr_param=["lr", "kp"],
)

for noise in kp_data_syn.values():
    for df in noise["dkl"].values():
        add_stats(df)
    for df in noise["asym"].values():
        add_stats(df)

## Plasticity noise

In [ ]:
kp_lr_plast = np.array([1e-5 * 2**i for i in [-3, -2, -1, 0, 1, 2, 3, 4]])
print(kp_lr_plast)

kp_data_plast = load_kp_runs(
    RESULTS_DIR / "plast_noise_kp_lr",
    noise_param=["stdp", "ws", "noise"],
    lr_param=["lr", "kp"],
)

for noise in kp_data_plast.values():
    for df in noise["dkl"].values():
        add_stats(df)
    for df in noise["asym"].values():
        add_stats(df)

## Plot

In [ ]:
fig, ax = plt.subplots(
    2, 2, sharex="col", figsize=(18 / 2.54, 11 / 2.54), tight_layout=True
)

ax[1, 0].set_xscale("log")
ax[1, 0].set_yscale("log")

asym = {
    0.4: dict(zip(("median", "err"), extract_data(kp_data_syn[0.4]["asym"]))),
    0.8: dict(zip(("median", "err"), extract_data(kp_data_syn[0.8]["asym"]))),
}
dkl = {
    0.4: dict(zip(("median", "err"), extract_data(kp_data_syn[0.4]["dkl"]))),
    0.8: dict(zip(("median", "err"), extract_data(kp_data_syn[0.8]["dkl"]))),
}
ax[0, 0].errorbar(
    kp_lr_syn,
    asym[0.4]["median"],
    asym[0.4]["err"],
    label=f"KP {STR_INIT_NOISE} = 0.4",
    marker="x",
    capsize=2,
    linestyle="--",
)
ax[0, 0].errorbar(
    kp_lr_syn,
    asym[0.8]["median"],
    asym[0.8]["err"],
    label=f"KP {STR_INIT_NOISE} = 0.8",
    marker="s",
    markerfacecolor="none",
    capsize=2,
    linestyle="--",
)
ax[1, 0].errorbar(
    kp_lr_syn,
    dkl[0.4]["median"],
    dkl[0.4]["err"],
    marker="x",
    capsize=2,
    linestyle="--",
)
ax[1, 0].errorbar(
    kp_lr_syn,
    dkl[0.8]["median"],
    dkl[0.8]["err"],
    marker="s",
    markerfacecolor="none",
    capsize=2,
    linestyle="--",
)

BEST_DKL = 4
ax[1, 0].plot(
    kp_lr_syn[BEST_DKL],
    dkl[0.4]["median"][BEST_DKL],
    marker="o",
    markerfacecolor="none",
    markersize=12.0,
    color="red",
)
BEST_DKL = 5
ax[1, 0].plot(
    kp_lr_syn[BEST_DKL],
    dkl[0.8]["median"][BEST_DKL],
    marker="o",
    markerfacecolor="none",
    markersize=12.0,
    color="red",
)

xlims = ax[1, 0].get_xlim()
ax[1, 0].axhline(
    ref_syn_noise_sal[0.4][0],
    linestyle="-.",
    color="C0",
    label=f"SAL {STR_INIT_NOISE} = 0.4",
)
fill_between(
    ax[1, 0],
    xlims,
    [ref_syn_noise_sal[0.4][1]] * 2,
    [ref_syn_noise_sal[0.4][2]] * 2,
    color="C0",
    alpha=0.2,
)
ax[1, 0].axhline(
    ref_syn_noise_sal[0.8][0],
    linestyle=":",
    color="C1",
    label=f"SAL {STR_INIT_NOISE} = 0.8",
)
fill_between(
    ax[1, 0],
    xlims,
    [ref_syn_noise_sal[0.8][1]] * 2,
    [ref_syn_noise_sal[0.8][2]] * 2,
    color="C1",
    alpha=0.2,
)

ax[0, 0].set_title("synaptic noise", pad=45)
ax[1, 0].set_xlim(xlims)
hs, ls = ax[0, 0].get_legend_handles_labels()
h, l = ax[1, 0].get_legend_handles_labels()
ax[0, 0].legend(
    hs + h,
    ls + l,
    bbox_to_anchor=(0.0, 1.02, 1.0, 0.102),
    loc="lower left",
    ncol=2,
    mode="expand",
    borderaxespad=0.0,
)

ax[1, 0].set_xlabel("weight decay rate $\lambda$")
ax[1, 0].set_ylabel(r"$D_\mathrm{KL}(p \| p^*)$")
ax[0, 0].set_ylabel(r"$\mathrm{Var}\, (W_{ij} - W_{ji})$")

# Plasticity noise
ax[1, 1].set_xscale("log")
ax[1, 1].set_yscale("log")

asym = {
    0.4: dict(zip(("median", "err"), extract_data(kp_data_plast[0.4]["asym"]))),
    0.8: dict(zip(("median", "err"), extract_data(kp_data_plast[0.8]["asym"]))),
}
dkl = {
    0.4: dict(zip(("median", "err"), extract_data(kp_data_plast[0.4]["dkl"]))),
    0.8: dict(zip(("median", "err"), extract_data(kp_data_plast[0.8]["dkl"]))),
}
ax[0, 1].errorbar(
    kp_lr_plast,
    asym[0.4]["median"],
    asym[0.4]["err"],
    label=f"KP {STR_PLAST_NOISE} = 0.4",
    marker="x",
    capsize=2,
    linestyle="--",
)
ax[0, 1].errorbar(
    kp_lr_plast,
    asym[0.8]["median"],
    asym[0.8]["err"],
    label=f"KP {STR_PLAST_NOISE} = 0.8",
    marker="s",
    markerfacecolor="none",
    capsize=2,
    linestyle="--",
)
ax[1, 1].errorbar(
    kp_lr_plast,
    dkl[0.4]["median"],
    dkl[0.4]["err"],
    marker="x",
    capsize=2,
    linestyle="--",
)
ax[1, 1].errorbar(
    kp_lr_plast,
    dkl[0.8]["median"],
    dkl[0.8]["err"],
    marker="s",
    markerfacecolor="none",
    capsize=2,
    linestyle="--",
)

BEST_DKL = 1
ax[1, 1].plot(
    kp_lr_plast[BEST_DKL],
    dkl[0.4]["median"][BEST_DKL],
    marker="o",
    markerfacecolor="none",
    markersize=12.0,
    color="red",
)
BEST_DKL = 2
ax[1, 1].plot(
    kp_lr_plast[BEST_DKL],
    dkl[0.8]["median"][BEST_DKL],
    marker="o",
    markerfacecolor="none",
    markersize=12.0,
    color="red",
)

xlims = ax[1, 1].get_xlim()
ax[1, 1].axhline(
    ref_plast_noise_sal[0.4][0],
    linestyle="-.",
    color="C0",
    label=f"SAL {STR_PLAST_NOISE} = 0.4",
)
fill_between(
    ax[1, 1],
    xlims,
    [ref_plast_noise_sal[0.4][1]] * 2,
    [ref_plast_noise_sal[0.4][2]] * 2,
    color="C0",
    alpha=0.2,
)
ax[1, 1].axhline(
    ref_plast_noise_sal[0.8][0],
    linestyle=":",
    color="C1",
    label=f"SAL {STR_PLAST_NOISE} = 0.8",
)
fill_between(
    ax[1, 1],
    xlims,
    [ref_plast_noise_sal[0.8][1]] * 2,
    [ref_plast_noise_sal[0.8][2]] * 2,
    color="C1",
    alpha=0.2,
)

hs, ls = ax[0, 1].get_legend_handles_labels()
h, l = ax[1, 1].get_legend_handles_labels()
ax[0, 1].legend(
    hs + h,
    ls + l,
    bbox_to_anchor=(0.0, 1.02, 1.0, 0.102),
    loc="lower left",
    ncol=2,
    mode="expand",
    borderaxespad=0.0,
)
ax[0, 1].set_title("plasticity noise", pad=45)
ax[0, 1].set_xlim(xlims)

ax[1, 1].set_xlabel("weight decay rate $\lambda$")
ax[1, 1].set_ylabel(r"$D_\mathrm{KL}(p \| p^*)$")
ax[0, 1].set_ylabel(r"$\mathrm{Var}\, (W_{ij} - W_{ji})$")

In [ ]:
fig.savefig(FIG_DIR / "kp_lr.png", bbox_inches="tight", dpi=300)
fig.savefig(FIG_DIR / "kp_lr.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "kp_lr.svg", bbox_inches="tight")